# Design Patterns

## Observer

A shop wants to trigger several actions when an order is paid (send a receipt, update the stock…), without the payment code knowing about these actions.

Write an `EventBus` class with two methods:

- `subscribe(event, callback)`, which subscribes a function to an event;
- `publish(event, *args)`, which calls, with `args` and in subscription order, every function subscribed to this event.

Subscribe `send_receipt` and `update_stock` to the `"order_paid"` event, then publish it for order 42: both messages must be printed. Publishing an event without subscribers must do nothing.

In [ ]:
def send_receipt(order_id: int) -> None:
    print(f"Receipt sent for order {order_id}")


def update_stock(order_id: int) -> None:
    print(f"Stock updated for order {order_id}")

In [ ]:
# Your code here

### Solution

In [ ]:
from collections import defaultdict
from collections.abc import Callable


class EventBus:
    def __init__(self) -> None:
        self._subscribers: defaultdict[str, list[Callable[..., None]]] = defaultdict(
            list
        )

    def subscribe(self, event: str, callback: Callable[..., None]) -> None:
        self._subscribers[event].append(callback)

    def publish(self, event: str, *args: object) -> None:
        for callback in self._subscribers.get(event, []):
            callback(*args)


bus = EventBus()
bus.subscribe("order_paid", send_receipt)
bus.subscribe("order_paid", update_stock)
bus.publish("order_paid", 42)
bus.publish("order_cancelled", 42)  # No subscriber: nothing happens

The code that publishes the event depends on no subscriber: you add an action by subscribing a function, without changing the payment code.

## Iterator

Implement the iterator pattern for the `Tree` class with depth-first traversal (DFS).


In [ ]:
from dataclasses import dataclass
from typing import Self


@dataclass
class Tree[T]:
    value: T
    children: list[Self]


tree = Tree("a", [Tree("b", [Tree("c", [])]), Tree("d", [])])
# print(f"DFS : {list(tree)}")

### Solution

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass
from typing import Self


@dataclass
class Tree[T]:
    value: T
    children: list[Self]

    def __iter__(self) -> Iterator[T]:
        yield self.value
        for child in self.children:
            yield from iter(child)


# Option with an external class
class DepthFirstSearch[T](Iterator[T]):
    def __init__(self, tree: Tree[T]) -> None:
        self._iterator = self._iterate(tree)

    @classmethod
    def _iterate(cls, tree: Tree[T]) -> Iterator[T]:
        yield tree.value
        for child in tree.children:
            yield from cls._iterate(child)

    def __next__(self) -> T:
        return next(self._iterator)


@dataclass
class Tree[T]:
    value: T
    children: list[Self]

    def __iter__(self) -> Iterator[T]:
        return DepthFirstSearch(self)


tree = Tree("a", [Tree("b", [Tree("c", [])]), Tree("d", [])])
print(f"DFS : {list(tree)}")